In [1]:
import pandas as pd

df = pd.read_parquet(r'C:\Users\NT751\Desktop\천영진\리스테리아ML\data\listeria_body.parquet')

print("shape:", df.shape)
df.head()

shape: (676537, 23)


,lims_data_key,hazard_label,judge_result,src_route,data_source,expr_iem_nm,result_value,result_value_num,judge_stdr,judge_stdr_num,...,train_prdlst_nm,prduct_nm,mnfctur_cmpny_nm,mnfctur_de,date_basis,addr,addr_source,legal_dong_code,lat,lon
0,O000001_160100103_5_4,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,0,0.0,<NA>,NaN,...,기타기준규격외,"화이트초콜릿 65℃, 30min",(주)쵸코하우스,2016-03-11,의뢰일,"경기도 성남시 중원구 둔촌대로 560(상대원동, 벽산테크노피아911,912,906,...",제조사소재지,4113310500,37.433239,127.164383
1,O000001_160100300_4_4,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,음성,NaN,<NA>,NaN,...,기타기준규격외,열무(세척전),(주)해마루,2016-03-22,의뢰일,"경기도 평택시 안중읍 성해2길 63-66 (학현리, 1, 2층)",제조사소재지,4122025300,36.986122,126.931168
2,O000033_170101151_1_5,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,음성,NaN,<NA>,NaN,...,"정액,수정란,종란","볶음-해물,야채(21일째)",<NA>,2017-02-22,제조일,전라북도 고창군 부안면 복분자로 434-36,업소주소,5279043000,35.514168,126.674095
3,O000027_170101235_2_7,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,음성,NaN,<NA>,NaN,...,기타기준규격외,크런치킹샌드(외포장),<NA>,2017-01-20,제조일,광주광역시 북구 하서로 222(양산동),업소주소,2917012800,35.209167,126.879715
4,1471142_170100097_1_1092137,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,음성,NaN,<NA>,NaN,...,축산물,생고기2(삼정),<NA>,2017-03-07,제조일,전라북도 정읍시 정신로 22-12 (하북동),제조사소재지,5218012100,35.595753,126.869698


## (2026-10-01 추가) 로마자로 적힌 한국 주소 행을 국내(domestic)로 옮긴다

- `src_route` 는 DB 함수 `staging_data.build_export_qual` 에서 `h.geo_route AS src_route` 로 만든 값이고, `geo_route` 는 테이블정의서(`ready_data` 시트)에 "domestic(한글주소→법정동) / overseas(비한글→지오코딩). 기상 결합은 domestic 만"으로 적혀 있다. 즉 **주소에 한글이 있는지**로 갈린 값이라, 한국 주소가 로마자로 적힌 행은 overseas 로 들어가 있다.
- 이 행들은 국내 관측소 기상을 붙일 수 있는 위치이므로 `src_route` 를 domestic 으로 바꾼다(사용자 결정 2026-10-01). 원자료 파일 `data/listeria_body.parquet` 는 바꾸지 않고, 이 노트북이 저장하는 `X.parquet` 에서만 바뀐다.
- 찾는 방법 두 가지를 따로 계산해 **같은 행을 가리킬 때만** 옮긴다(다르면 멈춘다).
  1. 낱말: `addr` 에 KOREA, 도 이름, 광역시 이름, `-EUP`·`-MYEON` 이 들어 있는 행
  2. 좌표: `lat`·`lon` 이 domestic 행 좌표 중 가장 가까운 점에서 0.5도 이내인 행 (위도·경도 차이의 유클리드 거리)
- 옮긴 행의 `legal_dong_code` 는 원자료에 값이 없어 비어 있는 그대로 둔다(채우지 않음).
- 한계: 낱말에도 안 걸리고 좌표도 0.5도 넘게 떨어진 곳으로 찍힌 한국 주소가 있다면 찾지 못한다. 시·군 이름 약 170개와 도로명 꼴로 넓혀 찾았을 때도 추가 행은 없었다(작업기록 2026-10-01).

In [2]:
import numpy as np

KR_WORDS = (r"KOREA|GYEONGGI|GYEONGSANG|JEOLLA|CHUNGCHEONG|GANGWON|JEJU"
            r"|SEOUL|BUSAN|INCHEON|DAEGU|DAEJEON|GWANGJU|ULSAN|-EUP\b|-MYEON\b")
NEAR_DEG = 0.5

is_ov = (df["src_route"].astype(str) == "overseas").to_numpy()
addr_up = df["addr"].astype("string").str.upper()
print(f"overseas {is_ov.sum():,}행 중 addr 에 한글이 든 행: {int(addr_up[is_ov].str.contains('[가-힣]').sum())}")

# 방법 1 — 낱말
by_word = is_ov & addr_up.str.contains(KR_WORDS, regex=True).fillna(False).to_numpy()

# 방법 2 — 좌표: domestic 행 좌표 중 가장 가까운 점까지의 거리(도)
dom_xy = df.loc[~is_ov, ["lat", "lon"]].drop_duplicates().to_numpy(dtype="float64")
ov_xy = df.loc[is_ov, ["lat", "lon"]].drop_duplicates().reset_index(drop=True)
u = ov_xy.to_numpy(dtype="float64")
min_d = np.empty(len(u))
for i in range(0, len(u), 500):
    c = u[i:i + 500]
    min_d[i:i + 500] = np.sqrt(((c[:, None, :] - dom_xy[None, :, :]) ** 2).sum(-1)).min(axis=1)
ov_xy["dist_deg"] = min_d
dist = df[["lat", "lon"]].merge(ov_xy, on=["lat", "lon"], how="left")["dist_deg"].to_numpy()
by_coord = is_ov & (dist <= NEAR_DEG)

print(f"낱말로 찾은 행 {by_word.sum()} · 좌표로 찾은 행 {by_coord.sum()} · 둘 다 {int((by_word & by_coord).sum())}")
assert (by_word == by_coord).all(), "낱말과 좌표가 가리키는 행이 다름 — 목록을 확인한 뒤 진행"
print(f"좌표로 찾지 않은 overseas 행의 가장 가까운 domestic 좌표까지 거리 최솟값: {np.nanmin(dist[is_ov & ~by_coord]):.2f}도")

KR_ROMAN = by_word
moved = df.loc[KR_ROMAN, ["lims_data_key", "addr", "lat", "lon", "addr_source", "date_basis", "judge_result", "train_prdlst_nm"]]
display(moved.groupby(["addr", "addr_source", "date_basis", "train_prdlst_nm", "judge_result"], observed=True)
             .size().rename("행수").reset_index().sort_values("행수", ascending=False).reset_index(drop=True))

before = df["src_route"].value_counts().to_dict()
df.loc[KR_ROMAN, "src_route"] = "domestic"
print("src_route 바꾸기 전:", before, "→ 후:", df["src_route"].value_counts().to_dict())
print(f"옮긴 행 {int(KR_ROMAN.sum())} (검출 {int((moved['judge_result'] == '검출').sum())}건)")
MOVED_KEYS = moved["lims_data_key"].tolist()

overseas 47,642행 중 addr 에 한글이 든 행: 0


낱말로 찾은 행 78 · 좌표로 찾은 행 78 · 둘 다 78
좌표로 찾지 않은 overseas 행의 가장 가까운 domestic 좌표까지 거리 최솟값: 1.80도


,addr,addr_source,date_basis,train_prdlst_nm,judge_result,행수
0,"15, BIRYONG-RO 331BEON-GIL, HWADO-EUP, NAMYANG...",제조사소재지,제조일,수산가공식품류,불검출,35
1,"15,Biryong-ro 331beon-gil, Hwado-eup, Namyangj...",제조사소재지,제조일,수산가공식품류,불검출,26
2,"126-33, Daeyangsandan-ro 125beon-gil, Mokpo-si...",제조사소재지,제조일,수산가공식품류,불검출,4
3,"17, ANSONGEUP-GIL, CHEONGDO-EUP, CHEONGDO-GUN,...",해외제조업소,선적일,수산물,불검출,3
4,"15, BIRYONG-RO 331 BEON-GIL, HWADO-EUP, NAMYAN...",제조사소재지,제조일,수산가공식품류,불검출,2
5,"679-25, GUMPO-RO, GOCHON-EUP, GIMPO-SI, GYEONG...",해외제조업소,선적일,수산물,불검출,2
6,"400, NOKSAN SANEOB BUK-RO, GANGSEO-GU, BUSAN, ...",해외제조업소,선적일,수산물,불검출,2
7,"111,WONYANG-RO,SEO-GU,BUSAN,KOREA.",해외제조업소,선적일,수산물,불검출,1
8,"1F, BUSAN PLANT OF DONGWON INDUSTRIES, 147 WON...",해외제조업소,선적일,수산물,불검출,1
9,"6, EULSUKDO-DAERO 755BEON-GIL, SAHA-GU, BUSAN,...",해외제조업소,선적일,수산가공식품류,불검출,1


src_route 바꾸기 전: {'domestic': 628895, 'overseas': 47642} → 후: {'domestic': 628973, 'overseas': 47564}
옮긴 행 78 (검출 0건)


In [3]:
import pandas as pd

# 1. 엑셀 스키마 로드
schema_path = r'C:\Users\NT751\Desktop\천영진\테이블명세서(모델팀)\테이블정의서.xlsx'
schema = pd.read_excel(schema_path, sheet_name='ready_data', header=0)

# 필요한 컬럼 추출 및 결측치 제거
schema = schema[['컬럼', '설명']].dropna(subset=['컬럼'])

# 텍스트 양끝 공백 제거 (매칭률 향상)
schema['컬럼'] = schema['컬럼'].astype(str).str.strip()
schema['설명'] = schema['설명'].astype(str).str.strip()

# 엑셀 내 중복 컬럼명이 있을 경우 첫 번째 설명만 유지 (행 증폭 방지)
schema = schema.drop_duplicates(subset=['컬럼'], keep='first')

# 2. 데이터프레임 컬럼 추출 및 전처리
_df_columns = pd.DataFrame({'df_컬럼명': [str(col).strip() for col in df.columns]})

# 3. 매핑 수행
df_칼럼명 = _df_columns.merge(
    schema,
    left_on='df_컬럼명',
    right_on='컬럼',
    how='left'
).drop(columns=['컬럼']).rename(columns={'설명': '명세서_설명'})

# 4. 매칭 여부 플래그 생성
df_칼럼명['매칭여부'] = df_칼럼명['명세서_설명'].notna()

# 5. 결과 출력
display(df_칼럼명)

unmatched_cols = df_칼럼명.loc[~df_칼럼명['매칭여부'], 'df_컬럼명'].tolist()
print(f"총 컬럼 수: {len(df_칼럼명)}개 / 매칭 성공: {df_칼럼명['매칭여부'].sum()}개")
print('명세서 미매칭 컬럼:', unmatched_cols)

,df_컬럼명,명세서_설명,매칭여부
0,lims_data_key,PK. 소스별 조합 키,True
1,hazard_label,위해요소명 (10종),True
2,judge_result,목표변수. 검출 / 불검출. 병합 시 '하나라도 검출이면 검출',True
3,src_route,NaN,False
4,data_source,NaN,False
5,expr_iem_nm,시험항목명(대표행 기준). 농관원 행은 조사 구분(anals_se)이 들어간다,True
6,result_value,검출값(원본 문자열). 정량·정성이 섞여 있어 물질 간 직접 비교 불가,True
7,result_value_num,NaN,False
8,judge_stdr,NaN,False
9,judge_stdr_num,NaN,False


총 컬럼 수: 23개 / 매칭 성공: 18개
명세서 미매칭 컬럼: ['src_route', 'data_source', 'result_value_num', 'judge_stdr', 'judge_stdr_num']


In [4]:
_n = len(df)
_rows = []
for col in df.columns:
    s = df[col]
    top5 = s.value_counts(dropna=True).head(5)
    _rows.append({
        '컬럼명': col,
        '데이터타입': str(s.dtype),
        '결측 수': int(s.isna().sum()),
        '결측 비율(%)': round(s.isna().mean() * 100, 2),
        '유니크 값 수': int(s.nunique(dropna=True)),
        '최빈값 5개': [f'{v} ({c:,})' for v, c in top5.items()],
    })

df_컬럼요약 = pd.DataFrame(_rows)
# 명세서 설명도 함께 붙이기
df_컬럼요약 = df_컬럼요약.merge(
    df_칼럼명[['df_컬럼명', '명세서_설명']],
    left_on='컬럼명', right_on='df_컬럼명', how='left'
).drop(columns='df_컬럼명')

print(f'총 {_n:,} rows × {df.shape[1]} columns')
display(df_컬럼요약)

총 676,537 rows × 23 columns


,컬럼명,데이터타입,결측 수,결측 비율(%),유니크 값 수,최빈값 5개,명세서_설명
0,lims_data_key,string,0,0.00,676537,"[O000001_160100103_5_4 (1), O000001_160100300_...",PK. 소스별 조합 키
1,hazard_label,category,0,0.00,1,"[리스테리아 (676,537)]",위해요소명 (10종)
2,judge_result,category,0,0.00,2,"[불검출 (674,568), 검출 (1,969)]",목표변수. 검출 / 불검출. 병합 시 '하나라도 검출이면 검출'
3,src_route,category,0,0.00,2,"[domestic (628,973), overseas (47,564)]",NaN
4,data_source,category,0,0.00,1,"[식약처 (676,537)]",NaN
5,expr_iem_nm,category,0,0.00,4,"[리스테리아모노사이토제네스 (660,130), 리스테리아 모노사이토제네스 (15,7...",시험항목명(대표행 기준). 농관원 행은 조사 구분(anals_se)이 들어간다
6,result_value,string,0,0.00,100,"[음성 (376,777), 음성,음성,음성,음성,음성 (281,865), 불검출 (...",검출값(원본 문자열). 정량·정성이 섞여 있어 물질 간 직접 비교 불가
7,result_value_num,float32,672760,99.44,2,"[0.0 (3,775), 2.0 (2)]",NaN
8,judge_stdr,string,669264,98.92,17,"[n=5,c=0,m=0/25g (4,692), n=5, c=0, m=0/25g (1...",NaN
9,judge_stdr_num,float32,676469,99.99,5,"[100.0 (62), 20.0 (3), 25.0 (1), 1000.0 (1), 0...",NaN


데이터 확인 후 결측 98%이상 칼럼 삭제

In [5]:
# 결측 비율 98% 이상인 컬럼 삭제
missing_rate = df.isna().mean()
drop_cols = missing_rate[missing_rate >= 0.98].index.tolist()

print("삭제 대상 컬럼 수:", len(drop_cols))
print("삭제 대상 컬럼:", drop_cols)

df = df.drop(columns=drop_cols)

print(f"원래 컬럼 수: {df.shape[1] + len(drop_cols)} -> 삭제 후 컬럼 수: {df.shape[1]}")
display(df.head())

삭제 대상 컬럼 수: 3
삭제 대상 컬럼: ['result_value_num', 'judge_stdr', 'judge_stdr_num']
원래 컬럼 수: 23 -> 삭제 후 컬럼 수: 20


,lims_data_key,hazard_label,judge_result,src_route,data_source,expr_iem_nm,result_value,inspct_instt_code,prdlst_code,train_prdlst_code,train_prdlst_nm,prduct_nm,mnfctur_cmpny_nm,mnfctur_de,date_basis,addr,addr_source,legal_dong_code,lat,lon
0,O000001_160100103_5_4,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,0,O000001,Z3800100000000,Z3800000000000,기타기준규격외,"화이트초콜릿 65℃, 30min",(주)쵸코하우스,2016-03-11,의뢰일,"경기도 성남시 중원구 둔촌대로 560(상대원동, 벽산테크노피아911,912,906,...",제조사소재지,4113310500,37.433239,127.164383
1,O000001_160100300_4_4,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,음성,O000001,Z3800100000000,Z3800000000000,기타기준규격외,열무(세척전),(주)해마루,2016-03-22,의뢰일,"경기도 평택시 안중읍 성해2길 63-66 (학현리, 1, 2층)",제조사소재지,4122025300,36.986122,126.931168
2,O000033_170101151_1_5,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,음성,O000033,Z2700400000000,Z2700000000000,"정액,수정란,종란","볶음-해물,야채(21일째)",<NA>,2017-02-22,제조일,전라북도 고창군 부안면 복분자로 434-36,업소주소,5279043000,35.514168,126.674095
3,O000027_170101235_2_7,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,음성,O000027,Z3800100000000,Z3800000000000,기타기준규격외,크런치킹샌드(외포장),<NA>,2017-01-20,제조일,광주광역시 북구 하서로 222(양산동),업소주소,2917012800,35.209167,126.879715
4,1471142_170100097_1_1092137,리스테리아,불검출,domestic,식약처,리스테리아모노사이토제네스,음성,1471142,B01010100F1000,B0100000000000,축산물,생고기2(삼정),<NA>,2017-03-07,제조일,전라북도 정읍시 정신로 22-12 (하북동),제조사소재지,5218012100,35.595753,126.869698


In [6]:
_n = len(df)
_rows = []
for col in df.columns:
    s = df[col]
    top5 = s.value_counts(dropna=True).head(5)
    _rows.append({
        '컬럼명': col,
        '데이터타입': str(s.dtype),
        '결측 수': int(s.isna().sum()),
        '결측 비율(%)': round(s.isna().mean() * 100, 2),
        '유니크 값 수': int(s.nunique(dropna=True)),
        '최빈값 5개': [f'{v} ({c:,})' for v, c in top5.items()],
    })

df_컬럼요약 = pd.DataFrame(_rows)
# 명세서 설명도 함께 붙이기
df_컬럼요약 = df_컬럼요약.merge(
    df_칼럼명[['df_컬럼명', '명세서_설명']],
    left_on='컬럼명', right_on='df_컬럼명', how='left'
).drop(columns='df_컬럼명')

print(f'총 {_n:,} rows × {df.shape[1]} columns')
display(df_컬럼요약)

총 676,537 rows × 20 columns


,컬럼명,데이터타입,결측 수,결측 비율(%),유니크 값 수,최빈값 5개,명세서_설명
0,lims_data_key,string,0,0.00,676537,"[O000001_160100103_5_4 (1), O000001_160100300_...",PK. 소스별 조합 키
1,hazard_label,category,0,0.00,1,"[리스테리아 (676,537)]",위해요소명 (10종)
2,judge_result,category,0,0.00,2,"[불검출 (674,568), 검출 (1,969)]",목표변수. 검출 / 불검출. 병합 시 '하나라도 검출이면 검출'
3,src_route,category,0,0.00,2,"[domestic (628,973), overseas (47,564)]",NaN
4,data_source,category,0,0.00,1,"[식약처 (676,537)]",NaN
5,expr_iem_nm,category,0,0.00,4,"[리스테리아모노사이토제네스 (660,130), 리스테리아 모노사이토제네스 (15,7...",시험항목명(대표행 기준). 농관원 행은 조사 구분(anals_se)이 들어간다
6,result_value,string,0,0.00,100,"[음성 (376,777), 음성,음성,음성,음성,음성 (281,865), 불검출 (...",검출값(원본 문자열). 정량·정성이 섞여 있어 물질 간 직접 비교 불가
7,inspct_instt_code,category,0,0.00,270,"[O000858 (65,811), O000392 (59,286), O000473 (...",검사기관 코드. 농관원 행은 NULL
8,prdlst_code,category,0,0.00,1251,"[Z3800100000000 (291,813), C0322020100000 (33,...",원본 식품공전 품목코드. 농관원은 품목명 매칭 결과
9,train_prdlst_code,category,170,0.03,52,"[Z3800000000000 (291,813), C0318000000000 (107...","학습용 품목코드. C계열 lv3, 그 외 lv2"


추가삭제 (쓸데없음)
- hazard_label,(위해요소명 (10종))
- prduct_nm(제품명)  
- data_source (식약처 값 하나 가짐)
- expr_iem_nm(다 리스테리아)
- 데이터 타입이 텍스트형인 모든 변수.(트리 모델에 그대로 못 넣음. 베이스라인에서는 빼고, 필요하면 나중에 따로 가공)
  - 단, lims_data_key(PK. 소스별 조합 키)는 기상 등 다른 데이터와 조인할 때 필요하므로 남긴다 (모델 피처에서는 제외)
- inspct_instt_code(검사기관 코드): 예측 시점에는 어느 검사기관으로 갈지 모르므로 피처로 쓸 수 없다 (2026-10-01 교수님 지적 → 사용자 결정으로 제거)

In [7]:
extra_drop_cols = ['hazard_label', 'prduct_nm', 'data_source', 'expr_iem_nm',
                   'inspct_instt_code']   # 검사기관 코드 — 예측 시점에 알 수 없는 값 (2026-10-01 제거)
keep_cols = ['judge_result', 'lims_data_key']
text_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

drop_cols = sorted((set(extra_drop_cols) | set(text_cols)) - set(keep_cols))
drop_cols = [c for c in drop_cols if c in df.columns]

df = df.drop(columns=drop_cols, errors='ignore')

print("삭제된 컬럼 수:", len(drop_cols))
print("삭제된 컬럼:", drop_cols)
print("남은 컬럼 수:", df.shape[1])
display(df.head())

삭제된 컬럼 수: 8
삭제된 컬럼: ['addr', 'data_source', 'expr_iem_nm', 'hazard_label', 'inspct_instt_code', 'mnfctur_cmpny_nm', 'prduct_nm', 'result_value']
남은 컬럼 수: 12


,lims_data_key,judge_result,src_route,prdlst_code,train_prdlst_code,train_prdlst_nm,mnfctur_de,date_basis,addr_source,legal_dong_code,lat,lon
0,O000001_160100103_5_4,불검출,domestic,Z3800100000000,Z3800000000000,기타기준규격외,2016-03-11,의뢰일,제조사소재지,4113310500,37.433239,127.164383
1,O000001_160100300_4_4,불검출,domestic,Z3800100000000,Z3800000000000,기타기준규격외,2016-03-22,의뢰일,제조사소재지,4122025300,36.986122,126.931168
2,O000033_170101151_1_5,불검출,domestic,Z2700400000000,Z2700000000000,"정액,수정란,종란",2017-02-22,제조일,업소주소,5279043000,35.514168,126.674095
3,O000027_170101235_2_7,불검출,domestic,Z3800100000000,Z3800000000000,기타기준규격외,2017-01-20,제조일,업소주소,2917012800,35.209167,126.879715
4,1471142_170100097_1_1092137,불검출,domestic,B01010100F1000,B0100000000000,축산물,2017-03-07,제조일,제조사소재지,5218012100,35.595753,126.869698


In [8]:
_n = len(df)
_rows = []
for col in df.columns:
    s = df[col]
    top5 = s.value_counts(dropna=True).head(5)
    _rows.append({
        '컬럼명': col,
        '데이터타입': str(s.dtype),
        '결측 수': int(s.isna().sum()),
        '결측 비율(%)': round(s.isna().mean() * 100, 2),
        '유니크 값 수': int(s.nunique(dropna=True)),
        '최빈값 5개': [f'{v} ({c:,})' for v, c in top5.items()],
    })

df_컬럼요약 = pd.DataFrame(_rows)
# 명세서 설명도 함께 붙이기
df_컬럼요약 = df_컬럼요약.merge(
    df_칼럼명[['df_컬럼명', '명세서_설명']],
    left_on='컬럼명', right_on='df_컬럼명', how='left'
).drop(columns='df_컬럼명')

print(f'총 {_n:,} rows × {df.shape[1]} columns')
display(df_컬럼요약)

총 676,537 rows × 12 columns


,컬럼명,데이터타입,결측 수,결측 비율(%),유니크 값 수,최빈값 5개,명세서_설명
0,lims_data_key,string,0,0.00,676537,"[O000001_160100103_5_4 (1), O000001_160100300_...",PK. 소스별 조합 키
1,judge_result,category,0,0.00,2,"[불검출 (674,568), 검출 (1,969)]",목표변수. 검출 / 불검출. 병합 시 '하나라도 검출이면 검출'
2,src_route,category,0,0.00,2,"[domestic (628,973), overseas (47,564)]",NaN
3,prdlst_code,category,0,0.00,1251,"[Z3800100000000 (291,813), C0322020100000 (33,...",원본 식품공전 품목코드. 농관원은 품목명 매칭 결과
4,train_prdlst_code,category,170,0.03,52,"[Z3800000000000 (291,813), C0318000000000 (107...","학습용 품목코드. C계열 lv3, 그 외 lv2"
5,train_prdlst_nm,category,0,0.00,53,"[기타기준규격외 (291,813), 유가공품류 (107,022), 식육가공품 및 포...",학습용 품목명. 미분류 시 '기타'(0.31%)
6,mnfctur_de,datetime64[ms],0,0.00,3909,"[2025-03-05 00:00:00 (639), 2025-05-20 00:00:0...",제조일(원본). 결측이 많아 기준일은 fallback 을 사용
7,date_basis,category,0,0.00,3,"[의뢰일 (333,425), 제조일 (296,040), 선적일 (47,072)]",기준일 출처 (제조일 / 의뢰일 / 수거일)
8,addr_source,category,0,0.00,4,"[제조사소재지 (327,704), 업소주소 (292,719), 해외제조업소 (47,...",주소 출처. 제조사소재지 → 인허가대장 → 해외제조업소 → 업소주소 → 검사기관 순...
9,legal_dong_code,category,47642,7.04,3769,"[4377034000 (7,426), 4113310500 (7,255), 11500...",법정동 코드.


## (2026-10-01 추가) 변수 제거 — 좌표·법정동·품목명, 품목코드 결측 채움

실험 2 `EXP/2/1_데이터준비.ipynb` 의 "변수 제거 1"·"변수 제거 3" 을 이 노트북으로 옮겼다(사용자 지시 2026-10-01). 이제 실험 1·2 가 같은 `X.parquet` 에서 출발한다.

| 처리 | 대상 | 근거 |
|---|---|---|
| 변수 제거 1 | `lat`·`lon`·`legal_dong_code` 원값 | `docs/04_실험계획/성능향상_방안.md` §3-1 (장소를 외워 test 에서 무너짐) |
| 변수 제거 3 | `train_prdlst_nm` | 같은 문서 §3-3 — `train_prdlst_code` 와 1:1 이라 중복 |
| 결측 채움 | `train_prdlst_code` 결측 170행 → `'기타'` | 사용자 지시 2026-10-01 |

- **결측 채움 전 확인** (`scripts/check_train_prdlst_code_null.py`, `data/listeria_body.parquet`): `train_prdlst_code` 결측 170행은 모두 `train_prdlst_nm` 이 `'기타'` 이고, `train_prdlst_nm` 이 `'기타'` 인 170행은 모두 `train_prdlst_code` 가 결측이다. (코드, 이름) 쌍은 53개로 1:1 이다. 아래 셀이 같은 조건을 다시 확인하고, 다르면 멈춘다.
- 채우는 값 `'기타'` 는 품목코드가 아니라 `train_prdlst_nm` 의 값을 그대로 가져온 것이다(`train_prdlst_code` 에 원래 `'기타'` 라는 코드는 없다). 170행 모두 불검출이다.
- `lat`·`lon` 은 모델 피처에서만 뺀다. 기상 조인(`3_기상조인_후_학습.ipynb`)에는 좌표가 필요하므로, 그 노트북이 원자료 `data/listeria_body.parquet` 에서 `lims_data_key` 로 `lat`·`lon` 을 따로 읽는다.
- 원자료 파일 `data/listeria_body.parquet` 는 바꾸지 않는다.


In [ ]:
MODEL_DROP_COLS = ['lat', 'lon', 'legal_dong_code', 'train_prdlst_nm']

# 1) train_prdlst_code 결측 → '기타'. 결측 행과 train_prdlst_nm '기타' 행이 같은 행일 때만 채운다
code_na = df['train_prdlst_code'].isna()
nm_etc = df['train_prdlst_nm'].astype(str) == '기타'
print(f"train_prdlst_code 결측 {int(code_na.sum())}행 · train_prdlst_nm '기타' {int(nm_etc.sum())}행 · 둘 다 {int((code_na & nm_etc).sum())}행")
assert (code_na == nm_etc).all(), "train_prdlst_code 결측 행과 train_prdlst_nm '기타' 행이 다름 — 채우지 말고 확인"
assert '기타' not in df['train_prdlst_code'].cat.categories, "train_prdlst_code 에 이미 '기타' 라는 값이 있음"
df['train_prdlst_code'] = df['train_prdlst_code'].cat.add_categories('기타').fillna('기타')

# 2) train_prdlst_code 와 train_prdlst_nm 이 1:1 인지 확인한 뒤 품목명을 뺀다
pair = df[['train_prdlst_code', 'train_prdlst_nm']].astype(str).drop_duplicates()
print(f"(품목코드, 품목명) 쌍 {len(pair)}개 · 코드 {pair['train_prdlst_code'].nunique()}종 · 이름 {pair['train_prdlst_nm'].nunique()}종")
assert len(pair) == pair['train_prdlst_code'].nunique() == pair['train_prdlst_nm'].nunique(), '품목코드와 품목명이 1:1 이 아님'

# 3) 좌표·법정동 원값과 품목명 제거
df = df.drop(columns=MODEL_DROP_COLS)
print('제거:', MODEL_DROP_COLS)
print('남은 컬럼', df.shape[1], '개:', df.columns.tolist())
print('결측이 있는 컬럼(행 수):', df.isna().sum().loc[lambda s: s > 0].to_dict())


In [9]:
# X / y 분리 (저장은 아래 저장 셀에서)
# y: 행 키 lims_data_key + judge_result(타깃)
#   - y 에도 lims_data_key 를 넣어, 조인·정렬로 행 순서가 바뀌어도 X 와 키로 다시 맞출 수 있게 한다 (2026-09-28 사용자 결정)
# X: 나머지 전부. 아래 두 컬럼은 피처가 아니라 보조용으로 일단 포함
#   - lims_data_key: 행 키 (조인·X/y 대응 확인용) — 모델 학습 시 제외
#   - mnfctur_de: 시간순 분할 기준 — 모델 학습 시 제외
# result_value(정답 누수)는 추가삭제 셀에서 이미 삭제됨
KEY_COL = 'lims_data_key'
y_cols = ['judge_result']

y = df[[KEY_COL] + y_cols].copy()
X = df.drop(columns=y_cols).copy()

# 행 키가 비어 있거나 겹치지 않는지, X 와 y 의 키 순서가 같은지 확인
assert X[KEY_COL].notna().all() and X[KEY_COL].is_unique, 'lims_data_key 에 결측·중복이 있음'
assert X[KEY_COL].equals(y[KEY_COL]), 'X 와 y 의 lims_data_key 순서가 다름'

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X 컬럼:", X.columns.tolist())
print("y 컬럼:", y.columns.tolist())
assert 'inspct_instt_code' not in X.columns
assert not set(MODEL_DROP_COLS) & set(X.columns) and X['train_prdlst_code'].notna().all()
print("X 의 src_route:", X['src_route'].value_counts().to_dict(), "| 옮긴 행이 domestic 인가:",
      bool((X.loc[X[KEY_COL].isin(MOVED_KEYS), 'src_route'] == 'domestic').all()))

X shape: (676537, 11)
y shape: (676537, 2)
X 컬럼: ['lims_data_key', 'src_route', 'prdlst_code', 'train_prdlst_code', 'train_prdlst_nm', 'mnfctur_de', 'date_basis', 'addr_source', 'legal_dong_code', 'lat', 'lon']
y 컬럼: ['lims_data_key', 'judge_result']
X 의 src_route: {'domestic': 628973, 'overseas': 47564} | 옮긴 행이 domestic 인가: True


In [10]:
# judge_result: 검출=1, 불검출=0
print("변환 전:", y['judge_result'].value_counts(dropna=False).to_dict())

y['judge_result'] = y['judge_result'].astype(str).map({'검출': 1, '불검출': 0})
assert y['judge_result'].notna().all(), "검출/불검출 외 값 존재"
y['judge_result'] = y['judge_result'].astype('int8')

print("변환 후:", y['judge_result'].value_counts().to_dict())
print(f"양성 비율: {y['judge_result'].mean() * 100:.3f}%")

변환 전: {'불검출': 674568, '검출': 1969}


변환 후: {0: 674568, 1: 1969}
양성 비율: 0.291%


In [11]:
y

,lims_data_key,judge_result
0,O000001_160100103_5_4,0
1,O000001_160100300_4_4,0
2,O000033_170101151_1_5,0
3,O000027_170101235_2_7,0
4,1471142_170100097_1_1092137,0
...,...,...
676532,1471124_170704332_1_11,0
676533,O000024_160103149_1_5,0
676534,O000024_160105715_1_4,0
676535,O000031_170700207_1_5,0


In [12]:
from pathlib import Path
import pandas as pd

data_dir = Path(r"C:\Users\NT751\Desktop\천영진\리스테리아ML\data")
data_dir.mkdir(parents=True, exist_ok=True)

X_path = data_dir / "X.parquet"
y_path = data_dir / "y.parquet"

X.to_parquet(X_path, index=False)
y.to_parquet(y_path, index=False)

print(f"Saved X: {X_path} | shape={X.shape}")
print(f"Saved y: {y_path} | shape={y.shape}")

Saved X: C:\Users\NT751\Desktop\천영진\리스테리아ML\data\X.parquet | shape=(676537, 11)
Saved y: C:\Users\NT751\Desktop\천영진\리스테리아ML\data\y.parquet | shape=(676537, 2)
